# Generative AI 015 — Retrievers

A retriever turns a query into relevant `Document`s. Everything here runs with
**no API key**. MultiQuery and contextual compression are not installed in this
environment, so those two strategies are built from scratch — a few lines each.

| Part | What we check |
|---|---|
| A | the plain retriever returns **exactly** what `similarity_search` does |
| B | MMR: redundancy **0.392 → 0.049** as λ falls, at a cost in relevance |
| C | multi-query: coverage **1/3 → 3/3**, and the zero-score filler it inherits |
| D | contextual compression: **45%** of the retrieved text kept |
| E | a custom retriever from one method |

Needs `langchain-core`, `scikit-learn`, `numpy`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — The plain retriever

In [ ]:
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.runnables import Runnable
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfEmbeddings(Embeddings):
    """Offline stand-in for an embedding model. Compares WORDS, not meaning."""
    def __init__(self, corpus):
        self.v = TfidfVectorizer(stop_words="english").fit(corpus)
    def embed_documents(self, texts):
        return self.v.transform(texts).toarray().tolist()
    def embed_query(self, text):
        return self.v.transform([text]).toarray()[0].tolist()

texts = ["Chroma is a vector database for storing and searching embeddings.",
         "LangChain connects language models to tools and data.",
         "FAISS is a library for fast similarity search over vectors.",
         "Pinecone is a managed cloud vector database.",
         "A retriever fetches documents relevant to a query."]
store = InMemoryVectorStore(embedding=TfidfEmbeddings(texts))
store.add_documents([Document(page_content=t) for t in texts])

query = "What is a vector database used for?"
direct = store.similarity_search(query, k=2)
retriever = store.as_retriever(search_kwargs={"k": 2})
via = retriever.invoke(query)

print([d.page_content for d in direct] == [d.page_content for d in via])   # True
print(isinstance(retriever, Runnable))                                     # True

# Identical results. The wrapper earns its place two ways: a Runnable joins
# a chain with "|", and the retriever is where search STRATEGY lives - so you
# can change the strategy without touching the chain around it.

In [ ]:
assert [d.page_content for d in direct] == [d.page_content for d in via]
assert isinstance(retriever, Runnable)
print("identical, and a Runnable")

Same documents. The wrapper earns its place two ways: it joins a chain with
`|`, and it is where **search strategy** lives.

## Part B — MMR: relevant and different

In [ ]:
from itertools import combinations
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

CLIMATE = [
    ("Climate change is melting glaciers faster than ever recorded.", "glaciers"),
    ("Glaciers are melting because climate change warms the mountains.", "glaciers"),
    ("Climate change melts glaciers, and the melting glaciers raise seas.", "glaciers"),
    ("Mountain glaciers shrink each year as climate change continues.", "glaciers"),
    ("Climate change brings longer heatwaves to Indian cities.", "heat"),
    ("Heatwaves from climate change strain hospitals and power grids.", "heat"),
    ("Climate change policy needs carbon pricing and renewable energy.", "policy"),
    ("Governments set climate change targets for net zero emissions.", "policy"),
    ("Climate change makes the ocean more acidic, harming coral reefs.", "ocean"),
]
ctexts = [t for t, _ in CLIMATE]
topic = dict(CLIMATE)
emb = TfidfEmbeddings(ctexts)
cstore = InMemoryVectorStore(embedding=emb)
cstore.add_documents([Document(page_content=t) for t in ctexts])
query = "climate change and glaciers"

def describe(docs):
    vecs = np.array(emb.embed_documents([d.page_content for d in docs]))
    redundancy = np.mean([cosine_similarity([vecs[i]], [vecs[j]])[0][0]
                          for i, j in combinations(range(len(vecs)), 2)])
    relevance = np.mean(cosine_similarity([emb.embed_query(query)], vecs)[0])
    return len({topic[d.page_content] for d in docs}), redundancy, relevance

for lam in (1.0, 0.5, 0.25, 0.0):
    docs = cstore.as_retriever(search_type="mmr",
                               search_kwargs={"k": 3, "fetch_k": 9,
                                              "lambda_mult": lam}).invoke(query)
    t, red, rel = describe(docs)
    print(f"lambda={lam:<5} topics {t}  redundancy {red:.3f}  relevance {rel:.3f}")

# lambda=1.0   topics 1  redundancy 0.392  relevance 0.503   <- three glacier copies
# lambda=0.5   topics 2  redundancy 0.113  relevance 0.372
# lambda=0.25  topics 3  redundancy 0.049  relevance 0.283
# lambda=0.0   topics 3  redundancy 0.049  relevance 0.283
#
# As lambda falls the results spread across topics and stop repeating each
# other - and relevance falls with them. That is the trade you are choosing:
# lambda=1 is "only relevance", lambda=0 is "only difference".

In [ ]:
sim = cstore.as_retriever(search_kwargs={"k": 3}).invoke(query)
low = cstore.as_retriever(search_type="mmr",
                          search_kwargs={"k": 3, "fetch_k": 9, "lambda_mult": 0.0}).invoke(query)
t_sim, red_sim, rel_sim = describe(sim)
t_low, red_low, rel_low = describe(low)

assert t_sim == 1 and t_low == 3          # one topic -> three
assert red_low < red_sim                  # less repetition
assert rel_low < rel_sim                  # ...and less relevance: the trade
print("plain search :", [topic[d.page_content] for d in sim])
print("mmr, lambda=0:", [topic[d.page_content] for d in low])

Plain search spent all three slots on glaciers saying the same thing. Lower λ
spreads the results — and lowers relevance. **λ = 1 is "only relevance", λ = 0
is "only difference".**

## Part C — Multi-query, and what it inherits

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

HEALTH = [
    ("A balanced diet with vegetables, pulses and whole grains supports health.", "diet"),
    ("Eating less sugar and processed food helps control weight.", "diet"),
    ("Regular exercise, even a daily walk, strengthens the heart.", "exercise"),
    ("Strength training twice a week protects muscles and bones.", "exercise"),
    ("Managing stress with sleep and meditation improves wellbeing.", "stress"),
    ("Chronic stress raises blood pressure and harms sleep.", "stress"),
    ("Solar panels store energy in batteries to keep the grid in balance.", "solar"),
    ("The solar system keeps its balance through gravity and orbital energy.", "solar"),
]
htopic = dict(HEALTH)
hstore = InMemoryVectorStore(embedding=TfidfEmbeddings([t for t, _ in HEALTH]))
hstore.add_documents([Document(page_content=t) for t, _ in HEALTH])
base = hstore.as_retriever(search_kwargs={"k": 3})

question = "How can I improve my energy and keep my life in balance?"
print([htopic[d.page_content] for d in base.invoke(question)])
# ['solar', 'solar', 'stress']   <- 'energy' and 'balance' matched; meaning did not

# The real MultiQueryRetriever asks an LLM for rewrites. CANNED here:
rewriter = FakeListChatModel(responses=[
    "What diet helps me stay healthy?\n"
    "How much exercise should I do to strengthen my heart?\n"
    "How do I manage stress and sleep better?"])
rewrites = rewriter.invoke(question).content.splitlines()

def multi_query(rewrites, min_score=None):
    merged, seen = [], set()
    for r in rewrites:
        for d, score in hstore.similarity_search_with_score(r, k=3):
            if min_score is not None and score <= min_score:
                continue
            if d.page_content not in seen:
                seen.add(d.page_content)
                merged.append(d)
    return merged

plain  = multi_query(rewrites)
strict = multi_query(rewrites, min_score=0.0)
for name, docs in (("plain union", plain), ("drop score 0", strict)):
    topics = [htopic[d.page_content] for d in docs]
    print(f"{name:<14} {len(docs)} docs, health topics "
          f"{len(set(topics) & {'diet','exercise','stress'})}/3, "
          f"off-topic {topics.count('solar')}")

# plain union    7 docs, health topics 3/3, off-topic 2
# drop score 0   5 docs, health topics 3/3, off-topic 0
#
# Multi-query fixed COVERAGE. It did not by itself remove the off-topic
# results - every solar document scored exactly 0.0000 and was only there
# because top-k always returns k. Top-k is a count, not a quality bar.

In [ ]:
# Where did the solar documents come from? Look at the actual scores.
for r in rewrites:
    print(f"{r[:44]:<46}",
          [f"{s:.4f} {htopic[d.page_content]}" for d, s in hstore.similarity_search_with_score(r, k=3)])

solar_scores = [s for r in rewrites for d, s in hstore.similarity_search_with_score(r, k=3)
                if htopic[d.page_content] == "solar"]
assert all(s == 0.0 for s in solar_scores)
print("\nevery solar result scored exactly 0.0 - it is filler, not a match")

**Top-k is a count, not a quality bar.** Ask for 3, get 3, relevant or not.
Multi-query fixed *coverage*; only the score threshold fixed *precision*.

> The rewrites were written by hand to stand in for an LLM. This measures what
> the merge does with good rewrites — not how good an LLM's rewrites are.

## Part D — Contextual compression

In [ ]:
MIXED_DOCS = [
    "The Grand Canyon is one of the most famous natural sites in the world. "
    "Photosynthesis is the process by which green plants convert sunlight into energy. "
    "Millions of tourists visit the canyon every year.",
    "Basketball was invented in 1891 by James Naismith. "
    "In photosynthesis, chlorophyll absorbs light and releases oxygen as a by-product. "
    "The NBA was founded in 1946.",
]
query = "What is photosynthesis?"
vec = TfidfVectorizer(stop_words="english").fit(
    [s for d in MIXED_DOCS for s in d.split(". ")] + [query])
qv = vec.transform([query])

def compress(doc):
    sentences = [s.strip().rstrip(".") + "." for s in doc.split(". ") if s.strip()]
    return [s for s in sentences if cosine_similarity(qv, vec.transform([s]))[0][0] > 0]

before = sum(len(d) for d in MIXED_DOCS)
kept = [s for d in MIXED_DOCS for s in compress(d)]
after = sum(len(s) for s in kept)
for s in kept:
    print(s)
print(f"{after} of {before} characters kept ({after/before:.0%})")

# Photosynthesis is the process by which green plants convert sunlight into energy.
# In photosynthesis, chlorophyll absorbs light and releases oxygen as a by-product.
# 162 of 363 characters kept (45%)
#
# Both documents matched because each CONTAINS a sentence about
# photosynthesis. Without compression the model would also get the Grand
# Canyon and the NBA. The real ContextualCompressionRetriever asks an LLM
# which parts matter; this version keeps sentences sharing a word with the
# query - fine here, and blind to an answer phrased differently.

In [ ]:
assert len(kept) == 2
assert all("photosynthesis" in s.lower() for s in kept)
assert after < before / 2
print(f"kept {after}/{before} characters")

## Part E — A custom retriever

In [ ]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun

class KeywordRetriever(BaseRetriever):
    """Returns documents containing every word of the query. Nothing clever."""
    documents: list[Document]
    k: int = 2

    def _get_relevant_documents(self, query: str, *,
                                run_manager: CallbackManagerForRetrieverRun):
        words = [w.lower().strip("?.,") for w in query.split()]
        hits = [d for d in self.documents
                if all(w in d.page_content.lower() for w in words)]
        return hits[: self.k]

r = KeywordRetriever(documents=[Document(page_content=t) for t, _ in HEALTH])
print([d.page_content[:40] for d in r.invoke("sleep stress")])
print(isinstance(r, Runnable))                                     # True
print([len(x) for x in r.batch(["sleep", "solar", "exercise"])])   # [2, 2, 1]

# One method written, invoke and batch inherited - the same pattern as the
# custom loader in lesson 012. It is how LangChain's twenty-odd retrievers
# all plug into the same chains.

## What to take away

- The plain retriever **equals** `similarity_search`; it exists to join chains
  and host strategy.
- **MMR** trades relevance for variety — measured both ways.
- **Multi-query** fixes coverage, not precision: **top-k is a count**.
- **Compression** sends only the sentences that answer.
- A custom retriever is **one method**.

## Exercises

1. Find the λ at which MMR first returns two topics. Is the change gradual or
   a jump? Why?
2. Replace the zero threshold in multi-query with a percentile of all scores.
   What goes wrong on a query where *nothing* is relevant?
3. Break the compressor: write a relevant sentence that shares no word with
   "What is photosynthesis?" and watch it get dropped.
4. Combine them: multi-query, then MMR over the merged pool, then
   compression. Measure characters and topics at each stage.
5. Write a retriever that returns nothing unless the best score clears a bar.
   When is returning nothing better than returning something?